In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

In [ ]:
class ProtT5Dataset(Dataset):
    def __init__(self, embeddings, scores):
        self.embeddings = torch.tensor(embeddings, dtype=torch.float32)
        self.scores = torch.tensor(scores, dtype=torch.float32)

    def __len__(self):
        return len(self.scores)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.scores[idx]

In [ ]:
def load_data_with_embeddings(csv_path, embeddings_path, indices_path, batch_size=32):
    """Load data using pre-existing split column."""
    df = pd.read_csv(csv_path)

    # Validate split column exists
    assert "split" in df.columns, "CSV must have a 'split' column with values: train/val/test"

    print(f"\nSplit distribution:")
    print(df["split"].value_counts().to_string())
    print(f"\nProteins per split:")
    print(df.groupby("split")["pdb_fn"].nunique().to_string())

    print(f"\nLoading embeddings from {embeddings_path}...")
    embeddings = np.load(embeddings_path, mmap_mode='r')
    print(f"Embeddings shape: {embeddings.shape}")

    print(f"Loading indices from {indices_path}...")
    embedding_indices = np.load(indices_path, mmap_mode='r')
    print(f"Indices shape: {embedding_indices.shape}")

    if len(embeddings) != len(embedding_indices):
        raise ValueError(
            f"Mismatch: embeddings have {len(embeddings)} samples, "
            f"indices have {len(embedding_indices)} samples"
        )

    # Build lookup: original_index → position in embeddings array
    index_to_position = {int(orig_idx): pos for pos, orig_idx in enumerate(embedding_indices)}

    missing = set(df["original_index"].values) - set(index_to_position.keys())
    if missing:
        raise ValueError(f"{len(missing)} CSV rows have no matching embedding: {list(missing)[:10]}")

    print(f"✓ All {len(df)} CSV rows have matching embeddings")

    def get_split_data(split_name):
        split_df = df[df["split"] == split_name].copy()
        positions = [index_to_position[int(i)] for i in split_df["original_index"].values]
        return embeddings[positions], split_df["total_score_z"].values

    train_emb, train_scores = get_split_data("train")
    val_emb,   val_scores   = get_split_data("val")
    test_emb,  test_scores  = get_split_data("test")

    train_loader = DataLoader(ProtT5Dataset(train_emb, train_scores), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(ProtT5Dataset(val_emb,   val_scores),   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(ProtT5Dataset(test_emb,  test_scores),  batch_size=batch_size, shuffle=False)

    print(f"\n✓ DataLoaders ready:")
    print(f"  Train: {len(train_emb)} samples")
    print(f"  Val:   {len(val_emb)} samples")
    print(f"  Test:  {len(test_emb)} samples")

    return train_loader, val_loader, test_loader, embeddings.shape[1]

In [ ]:
class MLPRegressionModel(nn.Module):
    def __init__(self, input_dim=1024, hidden_dims=[10], dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)


    def forward(self, x):
        return self.network(x).squeeze(-1)

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            all_preds.append(model(x).cpu().numpy())
            all_targets.append(y.cpu().numpy())

    preds   = np.concatenate(all_preds)
    targets = np.concatenate(all_targets)

    mse  = np.mean((preds - targets) ** 2)
    rmse = np.sqrt(mse)
    mae  = np.mean(np.abs(preds - targets))
    spearman_corr, _ = spearmanr(preds, targets)
    pearson_corr,  _ = pearsonr(preds, targets)
    ss_res = np.sum((targets - preds) ** 2)
    ss_tot = np.sum((targets - np.mean(targets)) ** 2)
    r2 = 1 - (ss_res / ss_tot)

    k = max(1, int(0.1 * len(targets)))
    top_k_overlap = len(set(np.argsort(targets)[-k:]) & set(np.argsort(preds)[-k:])) / k

    return {
        'mse': mse, 'rmse': rmse, 'mae': mae,
        'spearman': spearman_corr, 'pearson': pearson_corr,
        'r2': r2, 'top_10_overlap': top_k_overlap,
        'predictions': preds, 'targets': targets
    }

def print_metrics(metrics, prefix=""):
    print(f"{prefix}Metrics:")
    print(f"  MSE:              {metrics['mse']:.6f}")
    print(f"  RMSE:             {metrics['rmse']:.6f}")
    print(f"  MAE:              {metrics['mae']:.6f}")
    print(f"  Spearman ρ:       {metrics['spearman']:.6f}")
    print(f"  Pearson r:        {metrics['pearson']:.6f}")
    print(f"  R²:               {metrics['r2']:.6f}")
    print(f"  Top 10% overlap:  {metrics['top_10_overlap']:.2%}")

In [ ]:
def train(model, train_loader, val_loader, epochs=3, lr=1e-4,
          weight_decay=1e-3, device="cpu", eval_every=100, patience=10):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    history = {
        'train_steps': [], 'train_loss': [],
        'val_steps': [], 'val_mse': [], 'val_rmse': [], 'val_mae': [],
        'val_spearman': [], 'val_pearson': [], 'val_r2': [], 'val_top_10_overlap': []
    }

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    global_step = 0

    for epoch in range(epochs):
        model.train()
        print(f"\n{'='*70}\nEpoch {epoch+1}/{epochs}\n{'='*70}")

        epoch_train_loss, num_batches = 0.0, 0

        for batch_idx, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            preds = model(x)
            loss = loss_fn(preds, y)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            history['train_steps'].append(global_step)
            history['train_loss'].append(loss.item())
            epoch_train_loss += loss.item()
            num_batches += 1

            if batch_idx % 10 == 0:
                print(f"  Batch {batch_idx:4d} | Loss: {loss.item():.6f}")

            if global_step % eval_every == 0 and global_step > 0:
                metrics = evaluate(model, val_loader, device)
                history['val_steps'].append(global_step)
                for k in ['mse','rmse','mae','spearman','pearson','r2','top_10_overlap']:
                    history[f'val_{k}'].append(metrics[k])
                print(f"\n  [VALIDATION @ step {global_step}]")
                print(f"    RMSE:      {metrics['rmse']:.6f}")
                print(f"    Spearman:  {metrics['spearman']:.6f}")
                print(f"    Top 10%:   {metrics['top_10_overlap']:.2%}\n")
                model.train()

            global_step += 1

        # End-of-epoch evaluation
        avg_train_loss = epoch_train_loss / num_batches
        metrics = evaluate(model, val_loader, device)
        history['val_steps'].append(global_step)
        for k in ['mse','rmse','mae','spearman','pearson','r2','top_10_overlap']:
            history[f'val_{k}'].append(metrics[k])

        print(f"\n[END OF EPOCH {epoch+1}]")
        print(f"  Avg Train Loss: {avg_train_loss:.6f}")
        print_metrics(metrics, prefix="  ")

        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(metrics['mse'])
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr != old_lr:
            print(f"  📉 LR reduced: {old_lr:.2e} → {new_lr:.2e}")

        if metrics['mse'] < best_val_loss:
            best_val_loss = metrics['mse']
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0                    # reset only on genuine improvement
            print(f"  ✓ New best model (Val MSE: {best_val_loss:.6f})")
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{patience})")

            if patience_counter >= patience:        # 5 consecutive bad epochs → stop
                print(f"\n⚠ Early stopping triggered.")
                break

    if best_model_state:
        model.load_state_dict(best_model_state)
        print(f"\n✓ Restored best model")

    return history

In [ ]:
def plot_training(history):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    ax = axes[0, 0]
    window = 20
    if len(history['train_loss']) >= window:
        smoothed = np.convolve(history['train_loss'], np.ones(window)/window, mode='valid')
        ax.plot(history['train_steps'][window-1:], smoothed, label='Train (smoothed)', alpha=0.8, color='blue')
    ax.plot(history['train_steps'], history['train_loss'], alpha=0.2, color='blue', label='Train (raw)')
    ax.plot(history['val_steps'], history['val_mse'], marker='o', markersize=4, label='Val Loss', color='orange', linewidth=2)
    ax.set(xlabel='Steps', ylabel='MSE', title='Training & Validation Loss')
    ax.legend(); ax.grid(alpha=0.3)

    for ax, key, title, color in [
        (axes[0,1], 'val_rmse',          'Validation RMSE',                'blue'),
        (axes[0,2], 'val_mae',           'Validation MAE',                 'orange'),
        (axes[1,0], 'val_spearman',      'Validation Spearman ρ',          'green'),
        (axes[1,1], 'val_r2',            'Validation R²',                  'purple'),
        (axes[1,2], 'val_top_10_overlap','Top 10% Identification Accuracy', 'red'),
    ]:
        ax.plot(history['val_steps'], history[key], marker='o', markersize=4, color=color)
        ax.set(xlabel='Steps', title=title)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_predictions(metrics, title="Model Predictions"):
    preds, targets = metrics['predictions'], metrics['targets']
    k = max(1, int(0.1 * len(targets)))
    top_k = np.argsort(targets)[-k:]

    plt.figure(figsize=(10, 8))
    plt.scatter(targets, preds, alpha=0.3, s=20, label='All mutations', color='blue')
    plt.scatter(targets[top_k], preds[top_k], color='red', s=50, label='True Top 10%', alpha=0.7, edgecolors='darkred')
    lim = [min(targets.min(), preds.min()), max(targets.max(), preds.max())]
    plt.plot(lim, lim, 'k--', linewidth=2, label='Perfect prediction')
    plt.xlabel('True Score (Z-normalized)', fontsize=12)
    plt.ylabel('Predicted Score', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(fontsize=10)
    plt.grid(alpha=0.3)
    textstr = f"Spearman: {metrics['spearman']:.3f}\nR²: {metrics['r2']:.3f}\nTop 10%: {metrics['top_10_overlap']:.1%}"
    plt.text(0.05, 0.95, textstr, transform=plt.gca().transAxes, fontsize=11,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == "__main__":
    CSV_PATH        = "data_split_normalized_combined.csv"
    EMBEDDINGS_PATH = "prot_t5_embeddings_combined.npy"
    INDICES_PATH    = "prot_t5_indices_combined.npy"

    print("="*70)
    print("MLP MODEL - 3% SUBSET")
    print("="*70)

    train_loader, val_loader, test_loader, embedding_dim = load_data_with_embeddings(
        CSV_PATH, EMBEDDINGS_PATH, INDICES_PATH, batch_size=128
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nEmbedding dimension: {embedding_dim}")
    print(f"Using device: {device}")

    model = MLPRegressionModel(input_dim=embedding_dim, hidden_dims=[256, 64], dropout=0.5)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel parameters: {total_params:,}")

    history = train(
        model, train_loader, val_loader,
        epochs=50, lr=1e-4, weight_decay=1e-3,
        device=device, eval_every=200, patience=4
    )

    print("\n" + "="*70)
    print("FINAL TEST EVALUATION")
    print("="*70)
    test_metrics = evaluate(model, test_loader, device)
    print_metrics(test_metrics)

    plot_training(history)
    plot_predictions(test_metrics, title="MLP Test Set Predictions (3% Subset)")

    torch.save(model.state_dict(), "prot_t5_mlp_model_3pct.pt")
    print("\n✓ Model saved to: prot_t5_mlp_model_3pct.pt")